In [0]:
# Load bronze tables into spark dataframes
airlines_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airlines')
airports_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airports')
flights_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.flights')
cancellation_codes_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.cancellation_codes')

In [0]:
from pyspark.sql import functions as F

**1. Standardize data types**
- convert every value into the correct data format

In [0]:
# check schema of airlines_df
airlines_df.printSchema()

# check sample if it matches
display(airlines_df.limit(5))

In [0]:
# check schema of airports_df
airports_df.printSchema()

# check sample data of airports_df
display(airports_df.limit(5))

In [0]:
# check schema of cancellation_codes_df
cancellation_codes_df.printSchema()

# check sample data of cancellation_codes_df
display(cancellation_codes_df.limit(5))

In [0]:
# check schema of flights_df
flights_df.printSchema()

# check sample data of flights_df
display(flights_df.limit(5))

In [0]:
# covert columns to their corresponding appropriate data types
flights_df = flights_df.withColumn(
    'FLIGHT_NUMBER', F.col('FLIGHT_NUMBER').cast('string')
).withColumn(
    'DIVERTED', F.col('DIVERTED').cast('boolean')
).withColumn(
    'CANCELLED', F.col('CANCELLED').cast('boolean')
)

**2. Handle missing values**
- investigate and adress null entries

In [0]:
# During the basic data profiling, we already know which tables have null values
# Show rows with null values on the airports dataframe
display(airports_df.filter(
    F.col('LATITUDE').isNull()
))

In [0]:
# Create a mapping dataset for the missing lat and long values
# Data entered is gathered from Google Maps

missing_airport_coordinates = [
    ('ECP', 30.3582, -85.7956),
    ('PBG', 44.65094, -73.46814),
    ('UST', 29.959, -81.340)
]

# Turn into dataframe
airports_ref = spark.createDataFrame(missing_airport_coordinates, schema=['IATA_CODE', 'LAT_REF', 'LON_REF'])

# Perform join operation on the airports dataframe and missing_airport_coordinates
# Replace null values with the coalesced values to impute the nulls and then drop the reference columns created
airports_df = airports_df.join(
                    airports_ref,
                    on='IATA_CODE',
                    how='left'
                ).withColumn(
                    'LATITUDE',
                    F.coalesce(F.col('LATITUDE'), F.col('LAT_REF'))
                ).withColumn(
                    'LONGITUDE',
                    F.coalesce(F.col('LONGITUDE'), F.col('LON_REF'))
                ).drop(
                    'LAT_REF',
                    'LON_REF'
                )

In [0]:
flights_df.printSchema()

In [0]:
# Count each column's null on the flights dataframe
display(flights_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}') 
    for column in flights_df.columns]
))

In [0]:
# Create FLIGHT_DATE from year, month, day columns to help investigate missing SCHEDULED_TIME
flights_df = flights_df.withColumn(
    'FLIGHT_DATE',
    F.make_date(
        F.col('YEAR'),
        F.col('MONTH'),
        F.col('DAY')
    )
)

In [0]:
# Display the records with null on SCHEDULED_TIME

display(
        flights_df.filter(
        F.col('SCHEDULED_TIME').isNull()
    ).select(
        'FLIGHT_DATE',
        'ORIGIN_AIRPORT',
        'DESTINATION_AIRPORT',
        'SCHEDULED_DEPARTURE',
        'SCHEDULED_ARRIVAL',
        'SCHEDULED_TIME'
    )
)



In [0]:
# Display records from the flights dataframe with the same ORIGIN_AIRPORT, DESTINATION_AIRPORT, SCHEDULED_DEPARTURE, SCHEDULED_ARRIVAL
display(flights_df
 .filter(F.col('SCHEDULED_TIME').isNull())
 .alias('missing')
 .join(
     flights_df.alias('reference'),
     (
         (F.col('missing.ORIGIN_AIRPORT') == F.col('reference.ORIGIN_AIRPORT')) &
         (F.col('missing.DESTINATION_AIRPORT') == F.col('reference.DESTINATION_AIRPORT')) &
         (F.col('missing.SCHEDULED_DEPARTURE') == F.col('reference.SCHEDULED_DEPARTURE')) &
         (F.col('missing.SCHEDULED_ARRIVAL') == F.col('reference.SCHEDULED_ARRIVAL')) &
         (F.col('reference.SCHEDULED_TIME').isNotNull())
     ),
     'inner'
 )
 .select(
    F.col("missing.FLIGHT_DATE").alias("MISSING_DATE"),
    F.col("missing.ORIGIN_AIRPORT"),
    F.col("missing.DESTINATION_AIRPORT"),
    F.col("missing.SCHEDULED_DEPARTURE"),
    F.col("missing.SCHEDULED_ARRIVAL"),
    F.col("reference.FLIGHT_DATE").alias("REFERENCE_DATE"),
    F.col("reference.SCHEDULED_TIME").alias("REFERENCE_SCHEDULED_TIME")
)
 .orderBy('MISSING_DATE', 'REFERENCE_DATE')
)

In [0]:
# Create a mapping dataset for the missing SCHEDULED_TIME values based on the reference SCHEDULED_TIME
missing_scheduled_time = [
    ('2015-02-01', 172),
    ('2015-02-10', 172),
    ('2015-04-20', 178),
    ('2015-04-26', 111),
    ('2015-05-09', 130),
    ('2015-05-10', 113)
]

# Turn mapping dataset into a dataframe
scheduled_time_reference = spark.createDataFrame(
    missing_scheduled_time, 
    ['FLIGHT_DATE', 'SCHEDULED_TIME_REF']
)

# Join dataframes using FLIGHT_DATE as the key
flights_df = flights_df.join(
    scheduled_time_reference,
    on='FLIGHT_DATE',
    how='left'
).withColumn(
    'SCHEDULED_TIME',
    F.coalesce('SCHEDULED_TIME', 'SCHEDULED_TIME_REF')
).drop(
    'SCHEDULED_TIME_REF'
)

Since many of the null values in the `flights` DataFrame occur in operational columns, investigate whether these nulls represent genuinely missing data or are structurally expected based on the flight's operational status.

In [0]:
# Check how many flights are cancelled

display(
    flights_df.groupBy(
        F.col('CANCELLED').alias('IS_CANCELLED')
    ).agg(
        F.count('*').alias('COUNT')
    )
)

In [0]:
# For cancelled flights, how many rows per column are null
cancelled_flights = flights_df.filter(
    F.col('CANCELLED') == 'true'
)

display(cancelled_flights.select(
    [(F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}'))
    for column in cancelled_flights.columns]
))

In [0]:
# Of 89,884 cancelled flights, only 86,153 are null on DEPARTURE_TIME and DEPARTURE_DELAY
# Investigate cancelled flights that departed

display(cancelled_flights.filter(
    F.col('DEPARTURE_TIME').isNotNull() |
    F.col('DEPARTURE_DELAY').isNotNull())
)

# Cross check if the records which have DEPARTURE_TIME values are the same records that have DEPARTURE_DELAY
display(
    cancelled_flights.filter(
        (F.col('DEPARTURE_TIME').isNotNull() &
        F.col('DEPARTURE_DELAY').isNull()) |
        (F.col('DEPARTURE_TIME').isNull() &
        F.col('DEPARTURE_DELAY').isNotNull())
))

In [0]:
# Among cancelled flights which departed, some had values on TAXI_OUT and WHEELS_OFF, investigate those rows further

display(
    cancelled_flights.filter(
        (F.col('TAXI_OUT').isNotNull()) |
        (F.col('WHEELS_OFF').isNotNull())
    )
)

# Cross check if the same records which had null values on TAXI_OUT are the same records with null on WHEELS_OFF
display(cancelled_flights.filter(
    (F.col('TAXI_OUT').isNotNull() & F.col('WHEELS_OFF').isNull()) |
    (F.col('TAXI_OUT').isNull() & F.col('WHEELS_OFF').isNotNull())
))


**Conclusions so far:**
- Cancelled flights do not necessarily mean the aircraft never left the gate.
- Some cancelled flights have `DEPARTURE_TIME`, indicating that they departed the gate before cancellation.
- Some cancelled flights progressed further and have `TAXI_OUT` and `WHEELS_OFF` values.
- `WHEELS_OFF` is the furthest populated operational stage observed among cancelled flights.
- Cancelled flights with `WHEELS_OFF` have null values for subsequent operational columns such as `AIR_TIME`, `WHEELS_ON`, `TAXI_IN`, `AARIVAL_TIME`, etc.

**BTS Documentation:**
According to the Bureau of Transportation Statistics, a flight may still be classified as **cancelled after wheels-off**. These cases are referred to as **fly returns**, where the aircraft takes off but subsequently returns.
- Therefore, null values in later operational columns for cancelled flights can be structurally expected and should not automatically be treated as missing errors.
- In addition, cancelled flights which does not already have an aircraft assigned prior to cancellation should record the flight's `TAIL_NUMBER` as null.
- These structural nulls should not be imputed simply to make the columns look complete.

In [0]:
# Investigate diverted flights
diverted_flights = flights_df.filter(
    F.col('DIVERTED') == 'true'
)

print(f'Diverted flights:', diverted_flights.count())
display(diverted_flights)


In [0]:
# Check how many nulls exist per column in diverted flights

display(
    diverted_flights.select(
        [
            F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
            for column in diverted_flights.columns
        ]
    )
)

In [0]:
# The number of null values on WHEELS_ON, TAXI_IN, and ARRIVAL_TIME is the same
# Check if WHEELS_ON, TAXI_IN, and ARRIVAL_TIME have identical null patterns across all records

diverted_flights.filter(
    (F.col("WHEELS_ON").isNull() != F.col("TAXI_IN").isNull()) |
    (F.col("WHEELS_ON").isNull() != F.col("ARRIVAL_TIME").isNull())
).count()

In [0]:
# Check for records that were diverted and cancelled
display(
    flights_df.filter(
        (F.col('DIVERTED') == 'true') & (F.col('CANCELLED') == 'true') 
    )
)

**Conclusions:**

The null values observed in the operational fields of diverted flights are structural nulls rather than missing data errors.

`ELAPSED_TIME`, `AIR_TIME`, and `ARRIVAL_DELAY` are null for all diverted flights in the dataset. Meanwhile, `WHEELS_ON`, `TAXI_IN`, `ARRIVAL_TIME`, and other related fields are populated for approximately 80% of diverted flights and are null for the remaining ~20%. The null values in these three columns were also confirmed to occur on the same records.

These null values represent operational information that is unavailable or not applicable due to the flight being diverted. Imputing them would introduce artificial information into the dataset. Therefore, they will be retained.

**BTS Documentation:**

BTS distinguishes between diverted flights that eventually reach their scheduled destination and those that do not. For diverted flights that do not reach the scheduled destination, standard arrival-related fields may be left blank. Diverted flights that eventually reach their scheduled destination may still contain values for fields such as `WHEELS_ON`, `TAXI_IN`, and `ARRIVAL_TIME`.

This behavior is consistent with the observed pattern in the dataset, where approximately 80% of diverted flights contain `WHEELS_ON`, `TAXI_IN`, and `ARRIVAL_TIME`, while the remaining ~20% have these fields as null.

In [0]:
normal_flights = flights_df.filter(
    (F.col('DIVERTED') == 'false') & (F.col('CANCELLED') == 'false')
)

print('Normal flights count: ', normal_flights.count())
display(normal_flights)

In [0]:
# Investigate nulls on normal_flights 

display(
    normal_flights.select(
        [F.count(
            F.when(
                F.col(column).isNull(), 1
            )
        ).alias(f'missing_{column}')
        for column in normal_flights.columns]
    )
)

In [0]:
# Investigate non-null valueson the *_DELAY columns

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNotNull() |
        F.col('SECURITY_DELAY').isNotNull() |
        F.col('AIRLINE_DELAY').isNotNull() |
        F.col('LATE_AIRCRAFT_DELAY').isNotNull() |
        F.col('WEATHER_DELAY').isNotNull()
    )
)

In [0]:
# AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, and WEATHER_DELAY have the exact same amount of null values
# Additionally, these columns can can have a value that is 0
# Investigate if the null values are from the same records

display(
    normal_flights.filter(
        (
            F.col('AIR_SYSTEM_DELAY').isNull() |
            F.col('SECURITY_DELAY').isNull() |
            F.col('AIRLINE_DELAY').isNull() |
            F.col('LATE_AIRCRAFT_DELAY').isNull() |
            F.col('WEATHER_DELAY').isNull()
        )
        &
        (
            F.col('AIR_SYSTEM_DELAY').isNotNull() |
            F.col('SECURITY_DELAY').isNotNull() |
            F.col('AIRLINE_DELAY').isNotNull() |
            F.col('LATE_AIRCRAFT_DELAY').isNotNull() |
            F.col('WEATHER_DELAY').isNotNull()
        )
    )
)




In [0]:
# Since null values FROM *_DELAY columns are all from the same records, investigate these records

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNull()
    ).limit(30)
)

In [0]:
# Investigate other delay-related columns

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNull()
    ).select(
        F.min('DEPARTURE_DELAY').alias('MIN DEPT DELAY'),
        F.max('DEPARTURE_DELAY').alias('MAX DEPT DELAY'),
        F.min('ARRIVAL_DELAY').alias('MIN ARR DELAY'),
        F.max('ARRIVAL_DELAY').alias('MAX ARR DELAY')
    )
)

display(
    normal_flights.filter(
        F.col('AIR_SYSTEM_DELAY').isNotNull()
    ).select(
        F.min('DEPARTURE_DELAY').alias('MIN DEPT DELAY'),
        F.max('DEPARTURE_DELAY').alias('MAX DEPT DELAY'),
        F.min('ARRIVAL_DELAY').alias('MIN ARR DELAY'),
        F.max('ARRIVAL_DELAY').alias('MAX ARR DELAY')
    )
)

In [0]:
# There is a clear boundary between the ARRIVAL_DELAY of non-null and null *_DELAY columns, investigate further

display(
    normal_flights.filter(
        (F.col('AIR_SYSTEM_DELAY').isNull()) & (F.col('ARRIVAL_DELAY') >= 15)
    )
)

display(
    normal_flights.filter(
        (F.col('AIR_SYSTEM_DELAY').isNotNull()) & (F.col('ARRIVAL_DELAY') < 15)
    )
)


**Conclusions:**
- `AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `WEATHER_DELAY`, `AIRLINE_DELAY`, and `LATE_AIRCRAFT_DELAY` are null when `ARRIVAL_DELAY` is less than 15 minutes.
- For flights delayed 15 minutes or more, all five delay-cause columns are populated, including `0` when no delay is attributed to a particular cause.

**BTS Documentation:**
- On BTS reporting rules, causal delay data are required only for flights arriving at least 15 minutes late.
- Therefore, nulls in these columns for flights with `ARRIVAL_DELAY < 15` are structural and should not be imputed.

**3. Check for duplicate records**
- And remove, if necessary.

In [0]:
print('Airlines DataFrame')
print('Duplicate records:', airlines_df.count() - airlines_df.dropDuplicates().count())

print('\nAirports DataFrame')
print('Duplicate records:', airports_df.count() - airports_df.dropDuplicates().count())

print('\nCancellation Codes DataFrame')
print('Duplicate records:', cancellation_codes_df.count() - cancellation_codes_df.dropDuplicates().count())

print('\nFlights DataFrame')
print('Duplicate records:', flights_df.count() - flights_df.dropDuplicates().count())

**4. Create derived columns**
- Engineer useful columns that could be used for gold table

In [0]:
display(airports_df.limit(3))

Up until this point, datetime columns from flights_df are still in `int` data format. Creating a `datetime` column from these will help answer downstream analytical questions down the road.

In [0]:
display(flights_df.limit(3))

In [0]:
flights_df.withColumn(
    'SCHEDULED_DEPARTURE_DT',
    F.try_to_timestamp(
        F.concat_ws(
            ' ',
            F.col('FLIGHT_DATE'),
            F.lpad(
                F.col('SCHEDULED_DEPARTURE').cast('string'),
                4,
                '0'
            )
        ),
        F.lit('yyyy-MM-dd HHmm')
    )
)